# 03 — Visualizations
Loads the cleaned data from `02_data_clean_visualize_me_search.ipynb` and produces all maps.

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path

REPO_ROOT = Path().resolve().parent
os.chdir(REPO_ROOT)
print(os.getcwd())

## Load cleaned data

In [ ]:
# Load shapefile and cleaned data, then merge
nj_shapes = gpd.read_file('data/nj_zcta_2024.geojson')
final_df  = pd.read_csv('data/final_map.csv', dtype={'ZIPCODE': str})

map_df = nj_shapes.merge(final_df, on='ZIPCODE', how='left')

numeric_cols = [
    'max_pfas_ngl', 'mean_pfas_ngl', 'total_detections',
    'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic',
    'pct_native', 'pct_nonhispanic_white', 'total_pop', 'pct_minority'
]
for col in numeric_cols:
    map_df[col] = pd.to_numeric(map_df[col], errors='coerce')

map_df['max_exceeds_EPA']  = map_df['max_pfas_ngl'] > 4
map_df['mean_exceeds_EPA'] = map_df['mean_pfas_ngl'] > 4

print(f'Rows loaded: {len(map_df)}')
print(f'ZIPs exceeding EPA MCL (max):  {map_df["max_exceeds_EPA"].sum()}')
print(f'ZIPs exceeding EPA MCL (mean): {map_df["mean_exceeds_EPA"].sum()}')

## Max PFAS concentration by ZIP (ng/L)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

map_df.plot(
    column       = 'max_pfas_ngl',
    cmap         = 'OrRd',
    linewidth    = 0.5,
    edgecolor    = 'black',
    legend       = True,
    missing_kwds = {'color': 'lightgrey', 'label': 'No Data'},
    ax           = ax
)

ax.set_title('Max PFAS Levels by ZIP Code in New Jersey (ng/L)',
             fontsize=16, fontweight='bold', pad=20)
ax.set_axis_off()

plt.tight_layout()
plt.savefig('output/nj_pfas_map_max.png', dpi=300, bbox_inches='tight')
plt.show()

## Mean PFAS concentration by ZIP (ng/L)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

map_df.plot(
    column       = 'mean_pfas_ngl',
    cmap         = 'OrRd',
    linewidth    = 0.5,
    edgecolor    = 'black',
    legend       = True,
    missing_kwds = {'color': 'lightgrey', 'label': 'No Data'},
    ax           = ax
)

ax.set_title('Mean PFAS Levels by ZIP Code in New Jersey (ng/L)',
             fontsize=16, fontweight='bold', pad=20)
ax.set_axis_off()

plt.tight_layout()
plt.savefig('output/nj_pfas_map_mean.png', dpi=300, bbox_inches='tight')
plt.show()

## Binary map — ZIP codes exceeding EPA MCL (4 ng/L)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

cmap_binary = ListedColormap(['#d3d3d3', '#d73027'])

map_df.plot(
    column       = 'max_exceeds_EPA',
    cmap         = cmap_binary,
    linewidth    = 0.5,
    edgecolor    = 'black',
    legend       = False,
    missing_kwds = {'color': 'lightgrey'},
    ax           = ax
)

legend_elements = [
    Patch(facecolor='#d73027', edgecolor='black', label='Exceeds EPA MCL (> 4 ng/L)'),
    Patch(facecolor='#d3d3d3', edgecolor='black', label='Below EPA MCL'),
    Patch(facecolor='lightgrey', edgecolor='black', label='No Data'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=11)

ax.set_title('ZIP Codes Exceeding EPA PFAS Limit (4 ng/L) in New Jersey',
             fontsize=15, fontweight='bold', pad=20)
ax.set_axis_off()

plt.tight_layout()
plt.savefig('output/nj_pfas_exceeds_mcl.png', dpi=300, bbox_inches='tight')
plt.show()

## PFAS contamination vs race (side by side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 14))

# Left — mean PFAS
map_df.plot(
    column       = 'mean_pfas_ngl',
    cmap         = 'OrRd',
    linewidth    = 0.5,
    edgecolor    = 'grey',
    legend       = True,
    missing_kwds = {'color': 'lightgrey'},
    ax           = axes[0]
)
legend = axes[0].get_legend()
if legend is not None:
    legend.set_title('PFAS (ng/L)', prop={'size': 10})
axes[0].set_title('Mean PFAS Concentration by ZIP (ng/L)', fontsize=14, fontweight='bold')
axes[0].set_axis_off()

# Right — % minority
map_df.plot(
    column       = 'pct_minority',
    cmap         = 'Blues',
    linewidth    = 0.5,
    edgecolor    = 'grey',
    legend       = True,
    missing_kwds = {'color': 'lightgrey'},
    ax           = axes[1]
)
legend = axes[1].get_legend()
if legend is not None:
    legend.set_title('% Minority', prop={'size': 10})
axes[1].set_title('% Minority Population by ZIP', fontsize=14, fontweight='bold')
axes[1].set_axis_off()

plt.suptitle('PFAS Contamination vs Race in New Jersey',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/nj_pfas_vs_race_map.png', dpi=300, bbox_inches='tight')
plt.show()